In [1]:
import os
import pandas as pd
import numpy as np
from pathlib import Path
import plotly.express as px

root = Path("__file__").resolve().parents[4]
os.chdir(root)

In [2]:
tdf = pd.read_csv(root/"data"/"processed"/"gdelt_ohlcv_join.csv")

In [3]:
df = tdf.copy()
df.head()

,seendate,url,title,language,domain,socialimage,company,ticker,date,sentiment_score,sentiment_hits,sentiment_present,article_date,price_date,next_open,next_high,next_low,next_close,next_adj_close,next_volume
0,2026-01-04 01:15:00+00:00,https://finance.yahoo.com/news/stocks-early-ga...,Stocks Give Up Early Gains as Megacap Tech Sha...,English,finance.yahoo.com,https://s.yimg.com/ny/api/res/1.2/F_ZJDS12E4ol...,Apple,AAPL,2026-01-04,0.32,3.0,True,2026-01-04,2026-01-05,270.640015,271.51001,266.140015,267.26001,267.010162,45647200
1,2026-01-04 02:00:00+00:00,https://www.el-balad.com/6811251,Tim Cook Boosts Investment in Iconic Value Sto...,English,el-balad.com,https://www.el-balad.com/uploads/images/202601...,Apple,AAPL,2026-01-04,0.00,0.0,False,2026-01-04,2026-01-05,270.640015,271.51001,266.140015,267.26001,267.010162,45647200
2,2026-01-04 02:00:00+00:00,https://finance.yahoo.com/news/nike-ceo-direct...,"Nike CEO , Directors Signal Confidence in Comp...",English,finance.yahoo.com,https://s.yimg.com/ny/api/res/1.2/tc7o1tCMwKae...,Apple,AAPL,2026-01-04,0.76,1.0,True,2026-01-04,2026-01-05,270.640015,271.51001,266.140015,267.26001,267.010162,45647200
3,2026-01-04 02:00:00+00:00,https://www.thetimes.com/money/saving-investin...,How to invest well in 2026 six stock market st...,English,thetimes.com,https://www.thetimes.com/imageserver/image/%2F...,Apple,AAPL,2026-01-04,0.00,0.0,False,2026-01-04,2026-01-05,270.640015,271.51001,266.140015,267.26001,267.010162,45647200
4,2026-01-04 03:00:00+00:00,https://finance.yahoo.com/news/first-advantage...,Is First Advantage Stock a Buy After Onex Cana...,English,finance.yahoo.com,https://s.yimg.com/ny/api/res/1.2/V6AdfjZboZPB...,Apple,AAPL,2026-01-04,0.91,2.0,True,2026-01-04,2026-01-05,270.640015,271.51001,266.140015,267.26001,267.010162,45647200


In [4]:

df = df.copy()

df["article_date"] = pd.to_datetime(df["article_date"]).dt.floor("D")
df["price_date"] = pd.to_datetime(df["date"]).dt.floor("D")

daily = (
    df.groupby(["ticker","article_date"])["sentiment_score"]
      .agg(
          n_pos=lambda x: (x > 0).sum(),
          n_neg=lambda x: (x < 0).sum(),
          std=lambda x: x.std(),
          mean=lambda x: x.mean()
      )
)

daily["volume"] = daily["n_pos"] + daily["n_neg"]
daily["net_polarity"] = daily["n_pos"] - daily["n_neg"]
daily["net_polarity_ratio"] = daily["net_polarity"] / daily["volume"]

daily = daily.reset_index()
daily[daily["net_polarity_ratio"]<0]

,ticker,article_date,n_pos,n_neg,std,mean,volume,net_polarity,net_polarity_ratio
25,AMZN,2026-02-11,0,2,0.370842,-0.217143,2,-2,-1.000000
40,GOOGL,2026-02-03,1,6,0.428218,-0.194737,7,-5,-0.714286
41,GOOGL,2026-02-04,0,1,0.287253,-0.108571,1,-1,-1.000000
74,META,2026-02-13,3,4,0.554148,-0.044375,7,-1,-0.142857
92,MSFT,2026-02-16,2,3,0.674150,-0.145556,5,-1,-0.200000
109,TSLA,2026-01-04,1,5,0.319954,-0.091875,6,-4,-0.666667
111,TSLA,2026-01-11,4,5,0.561018,-0.010000,9,-1,-0.111111
117,TSLA,2026-02-03,0,1,0.380000,-0.190000,1,-1,-1.000000
127,TSLA,2026-02-16,1,2,0.350216,-0.038333,3,-1,-0.333333


In [98]:
returns = (
    df.groupby(["ticker","article_date"])
      .agg(
          next_open=("next_open","first"),
          next_close=("next_close","first")
      )
)

returns["next_ret"] = (returns["next_close"] - returns["next_open"]) / returns["next_open"]
returns["next_dir"] = (returns["next_ret"] > 0).astype(int)

In [6]:
analysis_df = daily.join(returns[["next_ret","next_dir"]], on=["ticker","article_date"])
analysis_df

,ticker,article_date,n_pos,n_neg,std,mean,volume,net_polarity,net_polarity_ratio,next_ret,next_dir
0,AAPL,2026-01-04,11,2,0.427985,0.194737,13,9,0.692308,-0.012489,0
1,AAPL,2026-01-11,12,3,0.598248,0.317308,15,9,0.600000,0.004206,1
2,AAPL,2026-01-12,16,3,0.490942,0.233953,19,13,0.684211,0.009006,1
3,AAPL,2026-01-15,16,2,0.497917,0.326970,18,14,0.777778,-0.009190,0
4,AAPL,2026-01-27,5,1,0.516527,0.270000,6,4,0.666667,-0.004696,0
...,...,...,...,...,...,...,...,...,...,...,...
123,TSLA,2026-02-12,3,2,0.357930,0.028929,5,1,0.200000,0.007555,1
124,TSLA,2026-02-13,4,3,0.555764,0.035000,7,1,0.142857,-0.004297,0
125,TSLA,2026-02-14,1,1,0.620537,0.000000,2,0,0.000000,-0.004297,0
126,TSLA,2026-02-15,4,1,0.501955,0.206154,5,3,0.600000,-0.004297,0


In [ ]:
analysis_df["net_polarity_ratio"].corr(analysis_df["next_dir"])

np.float64(0.015493990384418338)

In [8]:
analysis_df.groupby(
    analysis_df["net_polarity_ratio"] > 0
)["next_dir"].mean()

net_polarity_ratio
False    0.190476
True     0.439252
Name: next_dir, dtype: float64

In [9]:
analysis_df["next_dir"].mean()

np.float64(0.3984375)

In [10]:
def outliers(df_: pd.DataFrame, column: str):
    qs = (
        df_.groupby("ticker")[column]
        .quantile([0.25, 0.75])
        .unstack()
        .rename(columns={0.25: "q1", 0.75: "q3"})
    )
    qs["iqr"] = qs["q3"] - qs["q1"]
    qs[f"{column}_lower"] = qs["q1"] - (1.5 * qs["iqr"])
    qs[f"{column}_upper"] = qs["q3"] + (1.5 * qs["iqr"])
    df_2 = df_.join(qs[[f"{column}_lower", f"{column}_upper"]], on="ticker")
    outliers = df_2[ (df_2[column]<=df_2[f"{column}_lower"]) | (df_2[column]>=df_2[f"{column}_upper"]) ]
    return outliers

In [11]:
ratio_outlier = outliers(analysis_df, "net_polarity_ratio")
ratio_outlier

,ticker,article_date,n_pos,n_neg,std,mean,volume,net_polarity,net_polarity_ratio,next_ret,next_dir,net_polarity_ratio_lower,net_polarity_ratio_upper
11,AAPL,2026-02-16,4,3,0.555179,0.078824,7,1,0.142857,0.022593,1,0.255952,1.040079
25,AMZN,2026-02-11,0,2,0.370842,-0.217143,2,-2,-1.000000,-0.021377,0,-0.592105,1.421930
40,GOOGL,2026-02-03,1,6,0.428218,-0.194737,7,-5,-0.714286,-0.028925,0,-0.486174,1.451705
41,GOOGL,2026-02-04,0,1,0.287253,-0.108571,1,-1,-1.000000,0.060951,1,-0.486174,1.451705
117,TSLA,2026-02-03,0,1,0.380000,-0.190000,1,-1,-1.000000,-0.034367,0,-0.721429,1.392857


ratio is not fruitful. too little negative days. it honestly doesn't measure disagreement. its more directional.

In [15]:
std_outlier = outliers(analysis_df, "std")
std_outlier

,ticker,article_date,n_pos,n_neg,std,mean,volume,net_polarity,net_polarity_ratio,next_ret,next_dir,std_lower,std_upper
21,AMZN,2026-02-04,0,0,0.000000,0.000000,0,0,NaN,-0.009871,0,0.188063,0.775115
41,GOOGL,2026-02-04,0,1,0.287253,-0.108571,1,-1,-1.0,0.060951,1,0.306727,0.698759
65,META,2026-02-03,1,0,0.253333,0.084444,1,1,1.0,-0.027249,0,0.278521,0.729596
91,MSFT,2026-02-14,0,0,0.000000,0.000000,0,0,NaN,-0.005999,0,0.079966,0.784201
101,NVDA,2026-02-04,1,0,0.678823,0.480000,1,1,1.0,-0.017435,0,0.316797,0.671830
104,NVDA,2026-02-11,0,0,0.000000,0.000000,0,0,NaN,-0.031549,0,0.316797,0.671830
107,NVDA,2026-02-14,1,1,0.315880,0.012500,2,0,0.0,0.017885,1,0.316797,0.671830


In [18]:
mean_outliers = outliers(analysis_df, "mean")
mean_outliers

,ticker,article_date,n_pos,n_neg,std,mean,volume,net_polarity,net_polarity_ratio,next_ret,next_dir,mean_lower,mean_upper
29,GOOGL,2026-01-04,4,0,0.483422,0.621667,4,4,1.000000,-0.003526,0,-0.159314,0.540325
40,GOOGL,2026-02-03,1,6,0.428218,-0.194737,7,-5,-0.714286,-0.028925,0,-0.159314,0.540325
82,MSFT,2026-01-13,12,0,0.447312,0.451739,12,12,1.000000,-0.015178,0,-0.044591,0.307755
92,MSFT,2026-02-16,2,3,0.674150,-0.145556,5,-1,-0.200000,-0.005999,0,-0.044591,0.307755
121,TSLA,2026-02-09,8,2,0.699574,0.556000,10,6,0.600000,0.017054,1,-0.261532,0.474457


In [ ]:
mean_q_range=analysis_df["mean"].quantile([0.1,0.9]).reset_index(name="Mean Range")
lower,upper = mean_q_range["Mean Range"]

mean_q_outliers=analysis_df[
    (analysis_df["mean"]<=lower) |
    (analysis_df["mean"]>=upper)
]
display(mean_q_outliers)
mean_q_outliers.groupby(mean_q_outliers["mean"]>0.34)["next_dir"].mean()


,ticker,article_date,n_pos,n_neg,std,mean,volume,net_polarity,net_polarity_ratio,next_ret,next_dir
10,AAPL,2026-02-11,0,0,NaN,0.000000,0,0,NaN,-0.050292,0
18,AMZN,2026-01-16,1,1,0.760000,0.000000,2,0,0.000000,-0.011807,0
21,AMZN,2026-02-04,0,0,0.000000,0.000000,0,0,NaN,-0.009871,0
25,AMZN,2026-02-11,0,2,0.370842,-0.217143,2,-2,-1.000000,-0.021377,0
29,GOOGL,2026-01-04,4,0,0.483422,0.621667,4,4,1.000000,-0.003526,0
33,GOOGL,2026-01-11,5,0,0.490309,0.515556,5,5,1.000000,0.018600,1
40,GOOGL,2026-02-03,1,6,0.428218,-0.194737,7,-5,-0.714286,-0.028925,0
41,GOOGL,2026-02-04,0,1,0.287253,-0.108571,1,-1,-1.000000,0.060951,1
49,GOOGL,2026-02-13,10,3,0.693427,0.418667,13,7,0.538462,0.005326,1
54,META,2026-01-05,17,1,0.482230,0.384444,18,16,0.888889,0.001592,1


mean
False    0.200000
True     0.461538
Name: next_dir, dtype: float64

In [ ]:
mean_q_range

,index,Mean Range
0,0.1,0.000000
1,0.9,0.340548


In [44]:
analysis_df.drop(columns=(analysis_df.select_dtypes(include='str').columns)).corr()

,article_date,n_pos,n_neg,std,mean,volume,net_polarity,net_polarity_ratio,next_ret,next_dir
article_date,1.000000,-0.040282,0.302003,0.090807,-0.360295,0.069762,-0.211515,-0.359828,-0.006924,-0.056799
n_pos,-0.040282,1.000000,0.606073,0.327232,0.376320,0.964344,0.904422,0.205480,0.077318,0.106435
n_neg,0.302003,0.606073,1.000000,0.383005,-0.234327,0.794970,0.208792,-0.382485,0.091546,0.080176
std,0.090807,0.327232,0.383005,1.000000,0.281389,0.377367,0.196165,-0.135667,0.122776,0.093069
mean,-0.360295,0.376320,-0.234327,0.281389,1.000000,0.209048,0.588375,0.782124,0.078409,0.107575
volume,0.069762,0.964344,0.794970,0.377367,0.209048,1.000000,0.759261,0.028569,0.089430,0.107854
net_polarity,-0.211515,0.904422,0.208792,0.196165,0.588375,0.759261,1.000000,0.455589,0.045960,0.087858
net_polarity_ratio,-0.359828,0.205480,-0.382485,-0.135667,0.782124,0.028569,0.455589,1.000000,-0.075623,0.015494
next_ret,-0.006924,0.077318,0.091546,0.122776,0.078409,0.089430,0.045960,-0.075623,1.000000,0.769239
next_dir,-0.056799,0.106435,0.080176,0.093069,0.107575,0.107854,0.087858,0.015494,0.769239,1.000000


In [51]:
analysis_df["std"].corr(abs(analysis_df["next_ret"]))

np.float64(-0.13636453165744566)

slights higher with abs(next day)

In [ ]:
std_q_range=analysis_df.groupby(["ticker"])["std"].quantile([0.1,0.9]).reset_index(name="std_range")
display(std_q_range)

std_q_range=(std_q_range.pivot(
                    index="ticker",
                    columns="level_1",
                    values="std_range"
                ).rename(columns={0.1:"std_min",0.9:"std_max"})
                .reset_index()
)
std_q_range.columns.name = None
display(std_q_range)

analysis_df=analysis_df.merge(std_q_range,on="ticker")
analysis_df

# analysis_df.groupby("ticker")["std"]
# std_lower,std_upper = std_q_range["Std Range"]


# std_q_outliers=analysis_df[
#     (analysis_df["std"]<=std_lower) |
#     (analysis_df["std"]>=std_upper)
# ]
# display(std_q_outliers)

# std_q_outliers.groupby(std_q_outliers["std"]>0.34)["next_dir"].mean()

,ticker,level_1,std_range
0,AAPL,0.1,0.426186
1,AAPL,0.9,0.590807
2,AMZN,0.1,0.341821
3,AMZN,0.9,0.602815
4,GOOGL,0.1,0.429873
5,GOOGL,0.9,0.606154
6,META,0.1,0.388912
7,META,0.9,0.606484
8,MSFT,0.1,0.244654
9,MSFT,0.9,0.596950


,ticker,std_min,std_max
0,AAPL,0.426186,0.590807
1,AMZN,0.341821,0.602815
2,GOOGL,0.429873,0.606154
3,META,0.388912,0.606484
4,MSFT,0.244654,0.596950
5,NVDA,0.364744,0.586043
6,TSLA,0.356387,0.584738


,ticker,article_date,n_pos,n_neg,std,mean,volume,net_polarity,net_polarity_ratio,next_ret,next_dir,std_min,std_max
0,AAPL,2026-01-04,11,2,0.427985,0.194737,13,9,0.692308,-0.012489,0,0.426186,0.590807
1,AAPL,2026-01-11,12,3,0.598248,0.317308,15,9,0.600000,0.004206,1,0.426186,0.590807
2,AAPL,2026-01-12,16,3,0.490942,0.233953,19,13,0.684211,0.009006,1,0.426186,0.590807
3,AAPL,2026-01-15,16,2,0.497917,0.326970,18,14,0.777778,-0.009190,0,0.426186,0.590807
4,AAPL,2026-01-27,5,1,0.516527,0.270000,6,4,0.666667,-0.004696,0,0.426186,0.590807
...,...,...,...,...,...,...,...,...,...,...,...,...,...
123,TSLA,2026-02-12,3,2,0.357930,0.028929,5,1,0.200000,0.007555,1,0.356387,0.584738
124,TSLA,2026-02-13,4,3,0.555764,0.035000,7,1,0.142857,-0.004297,0,0.356387,0.584738
125,TSLA,2026-02-14,1,1,0.620537,0.000000,2,0,0.000000,-0.004297,0,0.356387,0.584738
126,TSLA,2026-02-15,4,1,0.501955,0.206154,5,3,0.600000,-0.004297,0,0.356387,0.584738


In [102]:
std_lower= analysis_df["std"]<analysis_df["std_min"]
std_upper= analysis_df["std"]>analysis_df["std_max"]


disagree_outs = analysis_df.loc[std_upper]
agree_outs = analysis_df.loc[std_lower]

display(len(disagree_outs))
display(len(disagree_outs[disagree_outs["volume"]>0]))
display(disagree_outs["next_dir"].mean())
display(disagree_outs["next_ret"].mean())
disagree_outs.groupby(["ticker",disagree_outs["volume"]==0])["next_ret"].mean()

# display(len(agree_outs))
# display(len(agree_outs[agree_outs["volume"]>0]))
# display(agree_outs["next_dir"].mean())
# agree_outs["next_ret"].mean()

15

15

np.float64(0.6)

np.float64(0.004279845663736618)

ticker  volume
AAPL    False     0.004206
AMZN    False     0.001731
GOOGL   False     0.003119
META    False     0.010884
MSFT    False     0.000657
NVDA    False     0.000225
TSLA    False     0.006379
Name: next_ret, dtype: float64

In [90]:
analysis_df.groupby(["ticker",analysis_df["volume"]==0])["next_ret"].mean()

ticker  volume
AAPL    False    -0.000028
        True     -0.050292
AMZN    False    -0.002111
        True     -0.009871
GOOGL   False     0.002383
META    False    -0.003335
MSFT    False    -0.004446
        True     -0.005999
NVDA    False    -0.005864
        True     -0.031549
TSLA    False    -0.002723
Name: next_ret, dtype: float64